## BMLL Notebook — Intraday Liquidity Profile

This notebook analyses **where** and **when** liquidity concentrates for an equity instrument
across all lit and dark venues. Using BMLL's `TRADES_PLUS` dataset — which classifies every
trade by execution mechanism — we build three complementary views:

| View | Question answered |
|---|---|
| **1-min bucket profile** | How does traded volume evolve minute-by-minute through the session? |
| **Price profile** | At which price levels is most of the day's volume transacted (market profile)? |
| **Price × 30-min heatmap** | Where do time *and* price concentration coincide? |

All three views can be split by **venue** (OPOL) — showing how each primary market
contributes to the ISIN's total liquidity — or by **trade classification** (BMLL_TRADE_TYPE),
distinguishing lit continuous, auctions, dark, and OTC flow.

A **7-day rolling average** overlay shows whether today's pattern is typical or anomalous.

This notebook illustrates the methods to compute and visualise intraday liquidity profiles in Snowflake.

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJRb9owFIX%2FSuQ9JzYho2AREIOxsXUrBdpqfTOOCW4dO%2FN1SPvv5yRFah9aaQ%2B2LPs79rk%2Bdzx9KlRwEhak0SnqRQQFQnOTSZ2n6Ga3DIcoAMd0xpTRIkXPAtB0MgZWqJLOKnfUG%2FG3EuACf5EG2h6kqLKaGgYSqGaFAOo43c5%2BXdI4IrS0xhluFHol%2BVjBAIR13uFZkoH09o7OlRTjuq6juh8Zm%2BOYEILJCHuqQT6d%2BSdf0zt8D5Ok4T3h8fWLty9Sd1%2Fwka19BwH9vtutw%2FXVdoeC2dnq3GioCmG3wp4kFzeby84AeAf7QqmwmZzgxwi0qQ%2BKPQpuirJy%2FsbIr%2FBBZFiZXPqiV4sUlY8y23wmbPBwKpL94FvC9%2Frux0O9N8fi9uvhfnNYX%2F8c2eU8ufpj72qOgttzqnGT6gqgEivdZOn8FokHIRmG5GLXG9F4SPskGsQX9yhY%2BCylZq5Vng0zzk2lHUS5MbkSrT%2BDm%2BBiLLNy6ofM0jlJ%2Bv3c1kPUNQht37ST%2Fyl7jF8rX%2Frst%2F%2F61WJtlOTPwdLYgrn3k%2BlFvXZHZuGhRakomFSzLLMCwCeklKnnVjDn29nZSiA86V5929CTfw%3D%3D&RelayState=ver%3A3-h

In [2]:
ticker = 'AZN'
mic = 'XLON'
date = '2025-09-08'

In [3]:
import warnings
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = None

## Parameters

Edit `TICKER` and `DATE` to change the instrument and session under analysis.
`LOOKBACK_DAYS` controls how many calendar days back to compute the rolling average
(fetches trading days only — weekends and holidays are excluded automatically).

In [4]:
TICKER        = "AZN"       # Ticker or ISIN
MIC           = "XLON"      # Primary exchange — tickers can collide across unrelated
                            # companies on different exchanges (e.g. "AZN" is also a
                            # US ADR and an unrelated Canadian shell company), so the
                            # exchange disambiguates which instrument is meant.
DATE          = "2025-09-08"
LOOKBACK_DAYS = 12          # calendar days back to look for ~7 trading days

# Derived date range for 7-day lookback
DATE_DT   = date.fromisoformat(DATE)
HIST_END  = DATE_DT - timedelta(days=1)
HIST_START = DATE_DT - timedelta(days=LOOKBACK_DAYS)

## ISIN Resolution

We look up the ISIN for the given ticker so we can pull data for **every venue**
that trades the same instrument (cross-venue analysis).

In [5]:
def _resolve_isin_sql(session, ref_table: str, ticker: str, mic: str, date_str: str) -> str:
    """Resolve ticker/ISIN to ISIN using SQL (Snowflake or Databricks).

    Filtered by exchange (MIC): tickers are not globally unique — e.g. "AZN"
    matches AstraZeneca's UK listing, its US ADR, and an unrelated Canadian
    shell company — so exchange + ticker is required to identify the
    instrument unambiguously.
    """
    d_back = (date.fromisoformat(date_str) - timedelta(days=7)).isoformat()
    sql = f"""
        SELECT DISTINCT ISIN
        FROM {ref_table}
        WHERE REFERENCE_DATE BETWEEN '{d_back}' AND '{date_str}'
          AND MIC = '{mic}'
          AND (UPPER(TICKER) = UPPER('{ticker}') OR UPPER(ISIN) = UPPER('{ticker}'))
        LIMIT 1
    """
    row = session.sql(sql).collect()
    if not row:
        raise ValueError(f"Cannot resolve '{ticker}' on MIC='{mic}' on {date_str}")
    return row[0]["ISIN"]


ISIN = _resolve_isin_sql(session, "REFERENCE", TICKER, MIC, DATE)
print(f"Resolved: {TICKER!r}@{MIC} -> ISIN {ISIN!r}")

Resolved: 'AZN'@XLON -> ISIN 'GB0009895292'


## Data Loading

The core query aggregates every trade for the ISIN (across all listings/venues) into
`(minute_of_day, price, venue, bmll_trade_type)` buckets.  This compact representation
supports all three view modes without re-querying the raw tick table.

In [6]:
def _load_intraday_notebook(isin: str, trade_date: str) -> pd.DataFrame:
    """
    Load intraday tick data for all listings of an ISIN via the BMLL API,
    then aggregate to (minute × price × venue × trade_type).
    """
    ref = reference.Reference(trade_date)
    listings = ref.get_listings(isin=isin)
    if listings.empty:
        raise ValueError(f"No listings found for ISIN={isin} on {trade_date}")

    frames = []
    for _, row in listings.iterrows():
        try:
            sec = NormalisedSecurity.from_listing_id(str(row["listing_id"]), trade_date)
            trades = sec.market_data().all_trades()
            trades["venue"] = row["opol"]
            frames.append(trades)
        except Exception as exc:
            print(f"  [skip] listing {row['listing_id']} / {row['opol']}: {exc}")

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    df = df.rename(columns={
        "trade_timestamp": "ts",
        "execution_size": "size",
        "bmll_trade_type": "bmll_trade_type",
    })
    df["minute_of_day"] = df["ts"].dt.hour * 60 + df["ts"].dt.minute
    df["notional"] = df["price"] * df["size"]
    return (
        df.groupby(["minute_of_day", "price", "venue", "bmll_trade_type"], as_index=False)
        .agg(volume=("size", "sum"), notional=("notional", "sum"), trade_count=("size", "count"))
    )


def _load_intraday_sql(conn_or_session, ref_table: str, trades_table: str,
                       isin: str, trade_date: str, is_spark: bool = False) -> pd.DataFrame:
    """Load intraday data via SQL (Snowflake or Databricks)."""
    d_back = (date.fromisoformat(trade_date) - timedelta(days=7)).isoformat()
    sql = f"""
        WITH listings AS (
            SELECT DISTINCT LISTING_ID, OPOL
            FROM {ref_table}
            WHERE REFERENCE_DATE BETWEEN '{d_back}' AND '{trade_date}'
              AND ISIN = '{isin}'
        )
        SELECT
            CAST(EXTRACT(HOUR FROM tp.TRADE_TIMESTAMP) AS INT) * 60
                + CAST(EXTRACT(MINUTE FROM tp.TRADE_TIMESTAMP) AS INT) AS "minute_of_day",
            tp.PRICE AS "price",
            l.OPOL AS "venue",
            tp.BMLL_TRADE_TYPE AS "bmll_trade_type",
            SUM(tp.SIZE) AS "volume",
            SUM(tp.PRICE * tp.SIZE) AS "notional",
            COUNT(*) AS "trade_count"
        FROM {trades_table} tp
        INNER JOIN listings l ON tp.LISTING_ID = l.LISTING_ID
        WHERE tp.TRADE_DATE = '{trade_date}'
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2
    """
    if is_spark:
        df = conn_or_session.sql(sql).toPandas()
    else:
        # Snowpark session
        df = conn_or_session.sql(sql).to_pandas()
    df.columns = [c.lower() for c in df.columns]
    return df

In [7]:
df_today = _load_intraday_sql(session, "REFERENCE", "PROD_L2_CBBO_DB.PUBLIC.TRADES_PLUS", ISIN, DATE)
print(f"Loaded {len(df_today):,} rows for {ISIN} on {DATE}")
print(f"Venues: {sorted(v for v in df_today['venue'].unique() if v is not None)}")
df_today.head()

Loaded 8,720 rows for GB0009895292 on 2025-09-08
Venues: ['XETR', 'XLON', 'XSTO']


,minute_of_day,price,venue,bmll_trade_type,volume,notional,trade_count
0,0,1512.7592,XSTO,SPECIAL_PRICE,733.0,1.108852e+06,1
1,0,1518.0000,XSTO,SPECIAL_PRICE,100.0,1.518000e+05,1
2,0,1518.5576,XSTO,SPECIAL_PRICE,312.0,4.737900e+05,1
3,0,1519.6732,XSTO,SPECIAL_PRICE,254.0,3.859970e+05,1
4,0,1519.7367,XSTO,SPECIAL_PRICE,435.0,6.610855e+05,1


In [8]:
def _load_history_sql(session, ref_table: str, trades_table: str,
                      isin: str, hist_start: date, hist_end: date) -> pd.DataFrame:
    """Load intraday data via SQL for a date range (7-day lookback)."""
    frames = []
    d = hist_start
    while d <= hist_end:
        try:
            df = _load_intraday_sql(session, ref_table, trades_table, isin, d.isoformat())
            if not df.empty:
                df["trade_date"] = d.isoformat()
                frames.append(df)
        except Exception:
            pass
        d += timedelta(days=1)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_hist = _load_history_sql(session, "REFERENCE", "PROD_L2_CBBO_DB.PUBLIC.TRADES_PLUS", ISIN, HIST_START, HIST_END)
n_days_hist = int(df_hist["trade_date"].nunique()) if not df_hist.empty else 0
print(f"Historical data: {n_days_hist} trading days, {len(df_hist):,} rows")

Historical data: 8 trading days, 73,946 rows


## Helper Utilities

In [9]:
def min_label(m: int | float) -> str:
    """Convert minute-of-day integer to HH:MM string."""
    m = int(m)
    return f"{m // 60:02d}:{m % 60:02d}"


def price_edges(df: pd.DataFrame, n_bins: int = 80) -> tuple[np.ndarray, list[float]]:
    """Compute n_bins evenly-spaced price bin edges and their centres."""
    p_min, p_max = float(df["price"].min()), float(df["price"].max())
    edges   = np.linspace(p_min, p_max, n_bins + 1)
    centers = [round((edges[i] + edges[i + 1]) / 2, 6) for i in range(n_bins)]
    return edges, centers

## View 1 — 1-min Bucket Profile

Plots total traded volume per minute, optionally stacked by venue or trade classification.
The dotted black line shows the rolling average for the same time slot.

In [10]:
METRIC     = "volume"         # "volume" | "notional" | "trade_count"
SPLIT_BY   = "venue"          # "venue"  | "bmll_trade_type" | None
SHOW_7D    = True

# ── aggregate ─────────────────────────────────────────────────────────────────
df_1m = df_today.copy()
df_1m["time"] = df_1m["minute_of_day"].apply(min_label)

group_cols = ["time", SPLIT_BY] if SPLIT_BY else ["time"]
agg_1m = df_1m.groupby(group_cols, sort=True)[METRIC].sum().reset_index()

# ── chart ─────────────────────────────────────────────────────────────────────
if SPLIT_BY:
    fig1 = px.bar(
        agg_1m, x="time", y=METRIC, color=SPLIT_BY,
        barmode="stack",
        title=f"1-min profile — {ISIN} — {DATE}",
        labels={"time": "Time (UTC)", METRIC: METRIC.replace("_", " ").title(), SPLIT_BY: SPLIT_BY},
        height=480,
    )
else:
    fig1 = px.bar(
        agg_1m, x="time", y=METRIC,
        title=f"1-min profile — {ISIN} — {DATE}",
        labels={"time": "Time (UTC)", METRIC: METRIC.replace("_", " ").title()},
        color_discrete_sequence=["#1f6feb"],
        height=480,
    )

if SHOW_7D and not df_hist.empty:
    dh = df_hist.copy()
    dh["time"] = dh["minute_of_day"].apply(min_label)
    avg_7d = (
        dh.groupby(["trade_date", "time"])[METRIC]
        .sum()
        .groupby("time")
        .mean()
        .reset_index()
    )
    fig1.add_trace(go.Scatter(
        x=avg_7d["time"], y=avg_7d[METRIC],
        mode="lines",
        name=f"{n_days_hist}-day avg",
        line=dict(color="black", width=2, dash="dot"),
    ))

fig1.update_layout(xaxis_tickangle=-45, legend=dict(orientation="h", y=1.05, x=0))
fig1.show()

## View 2 — Price Profile (Market Profile)

Volume-at-price histogram, read horizontally. The tallest bars (most volume) reveal
the "value area" — price levels where the market spent the most time trading.
An overlay of the 7-day average shows whether today's value area is stable or shifted.

In [11]:
N_BINS = 80

edges, centers = price_edges(df_today, N_BINS)
df_pp = df_today.copy()
df_pp["price_bin"] = pd.cut(df_pp["price"], bins=edges, labels=centers).astype(float)

group_cols = ["price_bin", SPLIT_BY] if SPLIT_BY else ["price_bin"]
agg_pp = df_pp.dropna(subset=["price_bin"]).groupby(group_cols)[METRIC].sum().reset_index()

if SPLIT_BY:
    fig2 = px.bar(
        agg_pp, x=METRIC, y="price_bin", color=SPLIT_BY,
        orientation="h", barmode="stack",
        title=f"Price profile — {ISIN} — {DATE}",
        labels={METRIC: METRIC.replace("_", " ").title(), "price_bin": "Price"},
        height=600,
    )
else:
    fig2 = px.bar(
        agg_pp, x=METRIC, y="price_bin",
        orientation="h",
        title=f"Price profile — {ISIN} — {DATE}",
        labels={METRIC: METRIC.replace("_", " ").title(), "price_bin": "Price"},
        color_discrete_sequence=["#1f6feb"],
        height=600,
    )

if SHOW_7D and not df_hist.empty:
    dh = df_hist.copy()
    dh["price_bin"] = pd.cut(dh["price"], bins=edges, labels=centers).astype(float)
    avg_7d_pp = (
        dh.dropna(subset=["price_bin"])
        .groupby(["trade_date", "price_bin"])[METRIC]
        .sum()
        .groupby("price_bin")
        .mean()
        .reset_index()
    )
    fig2.add_trace(go.Bar(
        x=avg_7d_pp[METRIC], y=avg_7d_pp["price_bin"],
        orientation="h",
        name=f"{n_days_hist}-day avg",
        marker_color="rgba(0,0,0,0.18)",
        opacity=0.75,
    ))

fig2.update_layout(legend=dict(orientation="h", y=1.05, x=0))
fig2.show()

## View 3 — Price × 30-min Heatmap

A 2-D heatmap with price on the Y-axis and 30-minute time buckets on the X-axis.
Darker cells indicate higher volume concentration. This view reveals patterns such as:
- Opening/closing auction spikes at specific price levels
- Intraday momentum where price and volume drift together
- Midday lulls at the previous session's close price

When a 7-day average is shown side-by-side, deviations from the typical pattern
(unusual price levels, atypical session timing) become immediately visible.

In [12]:
def _make_heatmap_pivot(df: pd.DataFrame, edges: np.ndarray, centers: list,
                        n_days_avg: int = 1) -> pd.DataFrame:
    df = df.copy()
    df["price_bin"]  = pd.cut(df["price"], bins=edges, labels=centers).astype(float)
    df["bucket_30m"] = df["minute_of_day"].apply(lambda m: min_label(int(m) - int(m) % 30))
    pivot = (
        df.dropna(subset=["price_bin"])
        .groupby(["bucket_30m", "price_bin"])["volume"]
        .sum()
        .div(n_days_avg)
        .reset_index()
        .pivot(index="price_bin", columns="bucket_30m", values="volume")
        .fillna(0)
        .sort_index(ascending=False)
    )
    return pivot


pivot_today = _make_heatmap_pivot(df_today, edges, centers)

if SHOW_7D and not df_hist.empty:
    n_days = df_hist["trade_date"].nunique()
    pivot_hist = _make_heatmap_pivot(df_hist, edges, centers, n_days_avg=n_days)
    # Align columns
    all_cols = sorted(set(pivot_today.columns) | set(pivot_hist.columns))
    pivot_today = pivot_today.reindex(columns=all_cols, fill_value=0)
    pivot_hist  = pivot_hist.reindex(columns=all_cols, fill_value=0)

    fig3 = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f"Today ({DATE})", f"{n_days}-day average"],
        horizontal_spacing=0.08,
    )
    y_labels = [f"{p:.4g}" for p in pivot_today.index.tolist()]

    fig3.add_trace(go.Heatmap(
        z=pivot_today.values, x=pivot_today.columns.tolist(), y=y_labels,
        colorscale="Blues", showscale=True, colorbar=dict(title="Volume", x=0.44),
        hovertemplate="Time: %{x}<br>Price: %{y}<br>Volume: %{z:,.0f}<extra></extra>",
    ), row=1, col=1)

    fig3.add_trace(go.Heatmap(
        z=pivot_hist.values, x=pivot_hist.columns.tolist(), y=y_labels,
        colorscale="Blues", showscale=True, colorbar=dict(title="Avg volume", x=1.0),
        hovertemplate="Time: %{x}<br>Price: %{y}<br>Avg volume: %{z:,.0f}<extra></extra>",
    ), row=1, col=2)

    fig3.update_layout(
        title=f"Price × 30-min heatmap — {ISIN}",
        height=540,
    )
    fig3.update_xaxes(tickangle=-45)
    fig3.update_yaxes(title_text="Price", row=1, col=1)

else:
    y_labels = [f"{p:.4g}" for p in pivot_today.index.tolist()]
    fig3 = go.Figure(go.Heatmap(
        z=pivot_today.values,
        x=pivot_today.columns.tolist(),
        y=y_labels,
        colorscale="Blues",
        colorbar=dict(title="Volume"),
        hovertemplate="Time: %{x}<br>Price: %{y}<br>Volume: %{z:,.0f}<extra></extra>",
    ))
    fig3.update_layout(
        title=f"Price × 30-min heatmap — {ISIN} — {DATE}",
        xaxis_title="Time (30-min bucket, UTC)",
        yaxis_title="Price",
        height=540,
        xaxis_tickangle=-45,
    )

fig3.show()

## Summary Statistics

Key metrics for the day's liquidity profile across all venues.

In [13]:
# VWAP and session summary
vwap = (df_today["price"] * df_today["volume"]).sum() / df_today["volume"].sum()
total_vol  = df_today["volume"].sum()
total_not  = df_today["notional"].sum()
n_venues   = df_today["venue"].nunique()
n_types    = df_today["bmll_trade_type"].nunique()

print(f"=== {ISIN}  {DATE} ===")
print(f"  Total volume   : {total_vol:>15,.0f} shares")
print(f"  Total notional : {total_not:>15,.2f}")
print(f"  Session VWAP   : {vwap:>15.4f}")
print(f"  Venues         : {n_venues}")
print(f"  Trade types    : {n_types}")
print()

# Volume by venue
print("Volume by venue:")
print(
    df_today.groupby("venue")["volume"]
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"{x:>12,.0f}  ({x / total_vol * 100:.1f}%)")
    .to_string()
)
print()

# Volume by classification
print("Volume by BMLL trade type:")
print(
    df_today.groupby("bmll_trade_type")["volume"]
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"{x:>12,.0f}  ({x / total_vol * 100:.1f}%)")
    .to_string()
)

=== GB0009895292  2025-09-08 ===
  Total volume   :       3,349,703 shares
  Total notional : 33,445,519,157.20
  Session VWAP   :       9984.6227
  Venues         : 3
  Trade types    : 8

Volume by venue:
venue
XLON       2,907,645  (86.8%)
XSTO         427,494  (12.8%)
XETR            9,583  (0.3%)

Volume by BMLL trade type:
bmll_trade_type
UNCROSSING                925,650  (27.6%)
LIT                       857,790  (25.6%)
OTC                       546,227  (16.3%)
SPECIAL_PRICE             516,937  (15.4%)
BENCHMARK_PRICE            254,440  (7.6%)
DARK                       183,089  (5.5%)
CLOSING_PRICE               59,589  (1.8%)
REQUEST_FOR_QUOTE            5,981  (0.2%)
